In [ ]:
# ============================================================
# Text Analysis Of Books Using Word Cloud
#
# Objectif : Analyser 3 livres de Lewis Carroll avec NLTK et spaCy
# pour faire du preprocessing, NER, POS tagging, word cloud, BoW et TF-IDF.
# ============================================================


# ============================================================
# Cellule 2 - Installation des bibliothèques
# ============================================================
!pip install -q nltk spacy wordcloud matplotlib scikit-learn
!python -m spacy download en_core_web_sm


# ============================================================
# Cellule 3 - Imports
# ============================================================
import requests
import re
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import matplotlib.pyplot as plt
import numpy as np

# Téléchargement des ressources NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

# Chargement du modèle spaCy
nlp = spacy.load('en_core_web_sm')


# ============================================================
# Text Preprocessing
# Nous allons charger 3 livres de Lewis Carroll depuis Project Gutenberg.
# ============================================================


# ============================================================
# Cellule 5 - Fonction load_texts()
# ============================================================
def load_texts():
    """
    Charge les 3 livres de Lewis Carroll depuis Project Gutenberg.
    Nettoie les crédits au début et à la fin.
    Retourne une liste de textes nettoyés.
    """
    urls = [
        "https://www.gutenberg.org/cache/epub/11/pg11.txt",      # Alice's Adventures in Wonderland
        "https://www.gutenberg.org/cache/epub/12/pg12.txt",      # Through the Looking-Glass
        "https://www.gutenberg.org/cache/epub/29042/pg29042.txt" # A Tangled Tale
    ]

    books = []
    for url in urls:
        response = requests.get(url)
        text = response.text

        # Nettoyage : suppression des crédits du début et de la fin
        # On cherche les marqueurs "START" et "END"
        start_marker = re.search(r'\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .+ \*\*\*', text)
        end_marker = re.search(r'\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .+ \*\*\*', text)

        if start_marker and end_marker:
            text = text[start_marker.end():end_marker.start()]

        books.append(text.strip())

    return books

# Chargement des livres
books = load_texts()
print(f"Nombre de livres chargés : {len(books)}")


# ============================================================
# Cellule 6 - Affichage des 200 premiers caractères
# ============================================================
# Affichage des 200 premiers caractères de chaque livre
for i, book in enumerate(books, 1):
    print(f"\n=== Livre {i} - 200 premiers caractères ===")
    print(book[:200])


# ============================================================
# Cellule 7 - Tokenisation
# ============================================================
# Tokenisation des textes
tokenized_books = []
for book in books:
    tokens = word_tokenize(book)
    tokenized_books.append(tokens)

# Affichage des 150 premiers tokens de chaque livre
for i, tokens in enumerate(tokenized_books, 1):
    print(f"\n=== Livre {i} - 150 premiers tokens ===")
    print(tokens[:150])


# ============================================================
# Cellule 8 - Suppression des stopwords
# ============================================================
stop_words = set(stopwords.words('english'))

filtered_books = []
for tokens in tokenized_books:
    filtered = [word.lower() for word in tokens if word.isalpha() and word.lower() not in stop_words]
    filtered_books.append(filtered)

# Vérification : compter les occurrences de quelques stopwords
print("\n=== Vérification suppression stopwords ===")
for i, tokens in enumerate(filtered_books, 1):
    count_the = tokens.count('the')
    count_a = tokens.count('a')
    count_and = tokens.count('and')
    print(f"Livre {i} : 'the' = {count_the}, 'a' = {count_a}, 'and' = {count_and}")


# ============================================================
# Cellule 9 - Stemming
# ============================================================
stemmer = PorterStemmer()

stemmed_books = []
for tokens in filtered_books:
    stemmed = [stemmer.stem(word) for word in tokens]
    stemmed_books.append(stemmed)

# Affichage des 50 premiers tokens stemmés
for i, stemmed in enumerate(stemmed_books, 1):
    print(f"\n=== Livre {i} - 50 premiers tokens stemmés ===")
    print(stemmed[:50])


# ============================================================
# Cellule 10 - Lemmatisation
# ============================================================
lemmatized_books = []
for book in books:
    # Limitation de la taille pour éviter les problèmes mémoire
    doc = nlp(book[:1000000])
    lemmatized = [token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop]
    lemmatized_books.append(lemmatized)

# Affichage des 50 premiers tokens lemmatisés
for i, lemmatized in enumerate(lemmatized_books, 1):
    print(f"\n=== Livre {i} - 50 premiers tokens lemmatisés ===")
    print(lemmatized[:50])


# ============================================================
# Cellule 11 - Comparaison stemming vs lemmatisation
# ============================================================
print("\n=== Comparaison stemming vs lemmatisation ===")
print("Stemming : réduit les mots à leur racine de manière brutale (règles heuristiques).")
print("Exemple : 'running' -> 'run', 'better' -> 'better'.")
print("\nLemmatisation : utilise un dictionnaire et la grammaire pour obtenir la forme canonique.")
print("Exemple : 'running' -> 'run', 'better' -> 'good'.")
print("\nLa lemmatisation est plus précise mais plus lente. Le stemming est plus rapide mais peut créer des formes inexistantes.")


# ============================================================
# Cellule 12 - POS Tagging
# ============================================================
print("\n=== POS Tagging ===")
for i, tokens in enumerate(tokenized_books, 1):
    pos_tags = nltk.pos_tag(tokens[:100])  # Premier cent tokens
    print(f"\nLivre {i} - POS tags (50 premiers) :")
    print(pos_tags[:50])


# ============================================================
# Cellule 13 - NER (Named Entity Recognition)
# ============================================================
print("\n=== Named Entity Recognition ===")
for i, tokens in enumerate(tokenized_books, 1):
    pos_tags = nltk.pos_tag(tokens)
    chunks = nltk.ne_chunk(pos_tags)

    entities = []
    for chunk in chunks:
        if hasattr(chunk, 'label'):
            entities.append((chunk.label(), ' '.join(c[0] for c in chunk)))

    print(f"\nLivre {i} - Entités nommées :")
    print(entities[:20])  # Affichage des vingt premières


# ============================================================
# Analysing The Text
# Visualisation et analyse des mots les plus fréquents avec word cloud et BoW.
# ============================================================


# ============================================================
# Cellule 15 - Word Clouds
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale"
]

for i, (book, ax, title) in enumerate(zip(filtered_books, axes, titles)):
    text = ' '.join(book)
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

    ax.imshow(wordcloud, interpolation='bilinear')
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()


# ============================================================
# Cellule 16 - Bag of Words - Top cinq mots
# ============================================================
all_texts = [' '.join(book) for book in filtered_books]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(all_texts)

# Somme des occurrences sur tous les documents
word_counts = bow_matrix.toarray().sum(axis=0)
words = vectorizer.get_feature_names_out()

# Top cinq mots
top_indices = word_counts.argsort()[-5:][::-1]
top_words = [(words[i], word_counts[i]) for i in top_indices]

print("\n=== Top cinq mots les plus fréquents (BoW) ===")
for word, count in top_words:
    print(f"{word}: {count}")


# ============================================================
# Cellule 17 - Affichage BoW
# ============================================================
print("\n=== Structure du BoW ===")
print(f"Nombre de documents : {bow_matrix.shape[0]}")
print(f"Nombre de mots uniques (vocabulaire) : {bow_matrix.shape[1]}")
print(f"\nExemple BoW - Document zéro (dix premiers mots) :")

for i, word in enumerate(words[:10]):
    print(f"Index {i} - Mot '{word}' : {bow_matrix[0, i]} occurrences")


# ============================================================
# Cellule 18 - Pie chart BoW
# ============================================================
labels = [f"{word}\n({count})" for word, count in top_words]
sizes = [count for _, count in top_words]

plt.figure(figsize=(8, 8))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
plt.title("Top cinq mots les plus fréquents (BoW)")
plt.show()


# ============================================================
# Cellule 19 - Analyse des résultats BoW
# ============================================================
print("\n=== Analyse des résultats ===")
print("Les mots les plus fréquents sont généralement des mots courants liés au contexte.")
print("Ils peuvent ne pas être très informatifs car ils apparaissent souvent dans tous les documents.")
print("Ces résultats sont attendus mais peu distinctifs entre les livres.")


# ============================================================
# Cellule 20 - TF-IDF
# TF-IDF permet d'identifier les mots importants pour chaque document
# en pondérant par leur rareté dans le corpus.
# ============================================================


# ============================================================
# Cellule 21 - BoW avec TF-IDF
# ============================================================
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(all_texts)

words_tfidf = tfidf_vectorizer.get_feature_names_out()

print("\n=== Top cinq mots par document (TF-IDF) ===")
for doc_idx in range(tfidf_matrix.shape[0]):
    doc_scores = tfidf_matrix[doc_idx].toarray()[0]
    top_indices = doc_scores.argsort()[-5:][::-1]
    top_words_doc = [(words_tfidf[i], doc_scores[i]) for i in top_indices]

    print(f"\nDocument {doc_idx + 1} ({titles[doc_idx]}) :")
    for word, score in top_words_doc:
        print(f"  {word}: {score:.4f}")


# ============================================================
# Cellule 22 - Pie charts TF-IDF
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for doc_idx, (ax, title) in enumerate(zip(axes, titles)):
    doc_scores = tfidf_matrix[doc_idx].toarray()[0]
    top_indices = doc_scores.argsort()[-5:][::-1]
    top_words_doc = [(words_tfidf[i], doc_scores[i]) for i in top_indices]

    labels = [f"{word}\n({score:.3f})" for word, score in top_words_doc]
    sizes = [score for _, score in top_words_doc]

    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title(f"{title}\n(TF-IDF Top cinq)")

plt.tight_layout()
plt.show()
